# Sprint 4: CNN-ViT Multi-label — 14 clases NIH

## Diferencia clave con Sprints anteriores

| | Sprints 1-3 | Sprint 4 |
|---|---|---|
| Tarea | Multi-clase (una clase por imagen) | **Multi-label (varias clases por imagen)** |
| Loss | CrossEntropyLoss | **BCEWithLogitsLoss** |
| Output | softmax (suma 1.0) | **sigmoid (independientes)** |
| Clases | 4 | **14** |
| Métricas | AUC macro | **AUC por clase + mAP** |
| Dataset | ~10k imágenes | **~70k imágenes** |

## Dataset
102,120 imágenes disponibles en HDF5. Sin restricción de balance.
El desbalance se maneja con `pos_weight` en BCEWithLogitsLoss.

**Runtime: T4 GPU. Tiempo estimado: ~3-4 horas (2 fases).**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ CPU'}")

H5_LOCAL        = '/content/nih_images.h5'
PROCESSED_ML    = '/content/processed_s4ml'
CLASS_NAMES_14  = [
    'Atelectasis','Cardiomegaly','Consolidation','Edema',
    'Effusion','Emphysema','Fibrosis','Hernia',
    'Infiltration','Mass','Nodule','Pleural_Thickening',
    'Pneumonia','Pneumothorax',
]


In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 2. Copiar HDF5 a disco local

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 4_000_000_000):
    size_gb = os.path.getsize(f'{NIH_DIR}/nih_images.h5') / (1024**3)
    print(f'Copiando HDF5 ({size_gb:.1f} GB)...')
    t0 = time.time()
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
    print(f'✓ Copiado en {(time.time()-t0)/60:.1f} min')

with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'✓ HDF5 local: {len(hf["images"]):,} imágenes')


## 3. Instalar módulos del Sprint 4 multi-label

In [ ]:
from pathlib import Path
content = r'''# configs/sprint4_multilabel.yaml
# Sprint 4 — CNN-ViT Multi-label (14 clases NIH)
# ─────────────────────────────────────────────────────────────
# Diferencias respecto a sprint4.yaml (4 clases):
#   - num_classes: 4 → 14
#   - loss: CrossEntropy → BCEWithLogitsLoss
#   - output: softmax (suma 1) → sigmoid (independientes)
#   - labels: un entero → vector binario de 14 posiciones
#   - métricas: AUC macro → AUC por clase + mAP
#   - datos: todos los pacientes sin restricción de balance estricto

project:
  name: "tesis-cxr-sprint4-multilabel"
  sprint: "4-multilabel"
  seed: 42

paths:
  project_root:   "/content/drive/MyDrive/Tesis_CXR"
  data_raw:       "/content/drive/MyDrive/Tesis_CXR/data/raw/nih"
  data_processed: "/content/drive/MyDrive/Tesis_CXR/data/processed_s4ml"
  experiments:    "/content/drive/MyDrive/Tesis_CXR/experiments_s4ml"
  checkpoints:    "/content/drive/MyDrive/Tesis_CXR/experiments_s4ml/checkpoints"
  logs:           "/content/drive/MyDrive/Tesis_CXR/experiments_s4ml/logs"

dataset:
  name: "nih_multilabel"
  csv_filename: "Data_Entry_subset_local.csv"
  views: ["PA", "AP"]

  # Las 14 patologías del NIH ChestX-ray14
  # "No Finding" = vector de ceros (no es una clase)
  classes:
    - {name: "Atelectasis",       index: 0}
    - {name: "Cardiomegaly",      index: 1}
    - {name: "Consolidation",     index: 2}
    - {name: "Edema",             index: 3}
    - {name: "Effusion",          index: 4}
    - {name: "Emphysema",         index: 5}
    - {name: "Fibrosis",          index: 6}
    - {name: "Hernia",            index: 7}
    - {name: "Infiltration",      index: 8}
    - {name: "Mass",              index: 9}
    - {name: "Nodule",            index: 10}
    - {name: "Pleural_Thickening",index: 11}
    - {name: "Pneumonia",         index: 12}
    - {name: "Pneumothorax",      index: 13}

  num_classes: 14

preprocessing:
  image_size: 224

splits:
  train_ratio: 0.70
  val_ratio:   0.15
  test_ratio:  0.15
  split_by:    "patient_id"
  # Multi-label: usar TODAS las imágenes en train (no 1/paciente)
  # Val/Test: 1 imagen/paciente para evaluación limpia
  all_images_in_train: true

model:
  backbone:    "densenet121-res224-nih"
  num_classes: 14
  pretrained:  true
  vit:
    embedding_dim: 512
    num_heads:     8
    num_layers:    4
    mlp_dim:       1024
    dropout:       0.1

training:
  batch_size:         32
  num_workers:        2
  use_amp:            true
  use_pos_weight:     true   # pos_weight para manejar desbalance

  phase1:
    epochs:        15
    learning_rate: 0.0001
    weight_decay:  0.0001
    patience:      5

  phase2:
    epochs:    40
    lr_cnn:    0.000001
    lr_vit:    0.00001
    weight_decay: 0.0001
    patience:  10
    scheduler:
      enabled: true
      type:    "CosineAnnealingLR"
      T_max:   40
      eta_min: 0.0000001

evaluation:
  # Multi-label: AUC por clase + mAP (no AUC macro de multi-clase)
  metrics: ["auc_per_class", "auc_macro", "map", "accuracy_per_class"]
  threshold: 0.5   # threshold por defecto para métricas binarias por clase
'''
p = Path(f'{CODE_DIR}/configs/sprint4_multilabel.yaml')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
from pathlib import Path
content = r'''"""
src/datasets/nih_multilabel.py

Dataset PyTorch para clasificación multi-label de 14 patologías NIH.

Diferencia clave respecto a nih_hdf5.py (multi-clase):
  - Retorna un vector binario de 14 posiciones en vez de un entero.
  - "No Finding" → vector de ceros [0,0,...,0].
  - Una imagen puede tener múltiples patologías activas.

Ejemplo:
  "Cardiomegaly|Effusion" → [0,1,0,0,1,0,0,0,0,0,0,0,0,0]
  "No Finding"             → [0,0,0,0,0,0,0,0,0,0,0,0,0,0]
  "Infiltration"           → [0,0,0,0,0,0,0,0,1,0,0,0,0,0]
"""
from __future__ import annotations

from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


# Las 14 patologías del NIH en orden canónico
CLASSES_14 = [
    "Atelectasis",        # 0
    "Cardiomegaly",       # 1
    "Consolidation",      # 2
    "Edema",              # 3
    "Effusion",           # 4
    "Emphysema",          # 5
    "Fibrosis",           # 6
    "Hernia",             # 7
    "Infiltration",       # 8
    "Mass",               # 9
    "Nodule",             # 10
    "Pleural_Thickening", # 11
    "Pneumonia",          # 12
    "Pneumothorax",       # 13
]

CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES_14)}
NUM_CLASSES  = 14


def findings_to_vector(finding_labels: str) -> np.ndarray:
    """
    Convierte el string de hallazgos del CSV a vector binario.

    Args:
        finding_labels: "Cardiomegaly|Effusion" o "No Finding"

    Returns:
        np.ndarray de shape (14,) con 0s y 1s.
    """
    vector = np.zeros(NUM_CLASSES, dtype=np.float32)

    if finding_labels.strip() == "No Finding":
        return vector  # todo ceros

    for finding in finding_labels.split("|"):
        finding = finding.strip()
        if finding in CLASS_TO_IDX:
            vector[CLASS_TO_IDX[finding]] = 1.0

    return vector


class NIHDatasetMultiLabel(Dataset):
    """
    Dataset multi-label que lee imágenes desde HDF5 y retorna
    vectores binarios de 14 posiciones como labels.

    Args:
        csv_path:  CSV con columnas Image Index, Finding Labels, Patient ID.
        h5_path:   Ruta al archivo nih_images.h5.
        transform: Pipeline de preprocesamiento.
    """

    def __init__(self, csv_path: str | Path,
                 h5_path:  str | Path,
                 transform=None):
        self.csv_path  = Path(csv_path)
        self.h5_path   = Path(h5_path)
        self.transform = transform

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV no encontrado: {self.csv_path}")
        if not self.h5_path.exists():
            raise FileNotFoundError(f"HDF5 no encontrado: {self.h5_path}")

        self.df = pd.read_csv(self.csv_path)
        self._validate()

        # Pre-computar vectores de labels para eficiencia
        self._labels = np.stack(
            [findings_to_vector(f) for f in self.df["Finding Labels"]],
            axis=0,
        )  # (N, 14) float32

        self._h5 = h5py.File(self.h5_path, "r")

    def _validate(self) -> None:
        required = {"Image Index", "Finding Labels", "Patient ID"}
        missing  = required - set(self.df.columns)
        if missing:
            raise ValueError(f"Columnas faltantes en CSV: {missing}")

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row        = self.df.iloc[idx]
        image_name = row["Image Index"]
        label_vec  = self._labels[idx]  # (14,) float32

        arr = self._h5["images"][image_name][:]  # (256, 256) uint8

        if self.transform is not None:
            arr = self.transform(arr)

        return {
            "image":       arr,
            "label":       torch.tensor(label_vec, dtype=torch.float32),
            "image_index": image_name,
            "finding_labels": row["Finding Labels"],
        }

    def class_distribution(self) -> dict:
        """Cuenta imágenes positivas por clase."""
        counts = self._labels.sum(axis=0).astype(int)
        return {CLASSES_14[i]: int(counts[i]) for i in range(NUM_CLASSES)}

    def compute_pos_weight(self, device: torch.device) -> torch.Tensor:
        """
        Calcula pos_weight para BCEWithLogitsLoss.
        pos_weight[i] = (N - N_pos[i]) / N_pos[i]

        Cuanto más rara es una clase, mayor es su peso.
        Esto compensa el desbalance sin necesidad de submuestreo.
        """
        N       = len(self._labels)
        n_pos   = self._labels.sum(axis=0).astype(float)
        n_pos   = np.maximum(n_pos, 1.0)  # evitar división por cero
        weights = (N - n_pos) / n_pos
        return torch.tensor(weights, dtype=torch.float32).to(device)

    def no_finding_count(self) -> int:
        """Imágenes sin ninguna patología."""
        return int((self._labels.sum(axis=1) == 0).sum())

    def __del__(self):
        if hasattr(self, "_h5") and self._h5.id.valid:
            self._h5.close()


def build_nih_multilabel_datasets(processed_dir: str,
                                   h5_path: str,
                                   train_transform=None,
                                   eval_transform=None) -> tuple:
    """Crea train/val/test datasets multi-label."""
    p = Path(processed_dir)
    h = Path(h5_path)
    return (
        NIHDatasetMultiLabel(p / "train.csv", h, train_transform),
        NIHDatasetMultiLabel(p / "val.csv",   h, eval_transform),
        NIHDatasetMultiLabel(p / "test.csv",  h, eval_transform),
    )


# ─── Smoke test ───────────────────────────────────────────────────
if __name__ == "__main__":
    print("Smoke test findings_to_vector:")

    cases = [
        ("No Finding",              [0]*14),
        ("Cardiomegaly",            None),
        ("Cardiomegaly|Effusion",   None),
        ("Infiltration|Pneumonia",  None),
    ]

    for finding, _ in cases:
        vec = findings_to_vector(finding)
        active = [CLASSES_14[i] for i in range(14) if vec[i] == 1]
        print(f"  '{finding}'")
        print(f"    → {vec.astype(int).tolist()}")
        print(f"    → Activas: {active if active else ['ninguna']}")
'''
p = Path(f'{CODE_DIR}/src/datasets/nih_multilabel.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
from pathlib import Path
content = r'''"""
src/evaluation/metrics_multilabel.py

Métricas para clasificación multi-label (Sprint 4 - 14 clases).

Métricas principales:
  - AUC por clase (one-vs-rest, estándar en NIH CXR papers)
  - AUC macro (promedio de las 14 clases)
  - mAP (mean Average Precision)
  - Accuracy por clase (con threshold 0.5)

Referencia: Wang et al. 2017 (ChestX-ray14) usa AUC por clase.
"""
from __future__ import annotations

import numpy as np
import torch
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    roc_curve,
)

CLASSES_14 = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion",    "Emphysema",    "Fibrosis",      "Hernia",
    "Infiltration","Mass",         "Nodule",         "Pleural_Thickening",
    "Pneumonia",   "Pneumothorax",
]


def compute_metrics_multilabel(y_true:  np.ndarray,
                                y_probs: np.ndarray,
                                threshold: float = 0.5) -> dict:
    """
    Calcula métricas multi-label.

    Args:
        y_true:  (N, 14) float32 — labels binarios ground truth.
        y_probs: (N, 14) float32 — probabilidades sigmoid del modelo.
        threshold: para calcular accuracy/precision/recall por clase.

    Returns:
        dict con auc_per_class, auc_macro, map, accuracy_per_class, etc.
    """
    y_true  = _to_numpy(y_true)
    y_probs = _to_numpy(y_probs)
    y_pred  = (y_probs >= threshold).astype(int)

    n_classes   = y_true.shape[1]
    auc_list    = []
    ap_list     = []
    acc_list    = []
    sens_list   = []
    spec_list   = []

    for c in range(n_classes):
        y_c    = y_true[:, c]
        p_c    = y_probs[:, c]
        pred_c = y_pred[:, c]

        n_pos = y_c.sum()
        n_neg = len(y_c) - n_pos

        # AUC — solo si hay positivos y negativos
        if n_pos > 0 and n_neg > 0:
            auc = float(roc_auc_score(y_c, p_c))
            ap  = float(average_precision_score(y_c, p_c))
        else:
            auc = float("nan")
            ap  = float("nan")

        auc_list.append(auc)
        ap_list.append(ap)

        # Accuracy, Sensitivity, Specificity con threshold
        tp = int(((pred_c == 1) & (y_c == 1)).sum())
        fp = int(((pred_c == 1) & (y_c == 0)).sum())
        tn = int(((pred_c == 0) & (y_c == 0)).sum())
        fn = int(((pred_c == 0) & (y_c == 1)).sum())

        acc  = (tp + tn) / (tp + fp + tn + fn) if (tp+fp+tn+fn) > 0 else 0.0
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        acc_list.append(float(acc))
        sens_list.append(float(sens))
        spec_list.append(float(spec))

    # Promedios excluyendo NaN
    valid_auc = [a for a in auc_list if not np.isnan(a)]
    valid_ap  = [a for a in ap_list  if not np.isnan(a)]

    return {
        "auc_per_class":      auc_list,          # lista de 14
        "auc_macro":          float(np.mean(valid_auc)) if valid_auc else float("nan"),
        "ap_per_class":       ap_list,            # lista de 14
        "map":                float(np.mean(valid_ap))  if valid_ap  else float("nan"),
        "accuracy_per_class": acc_list,
        "accuracy_macro":     float(np.mean(acc_list)),
        "sensitivity_per_class": sens_list,
        "sensitivity_macro":  float(np.mean(sens_list)),
        "specificity_per_class": spec_list,
        "specificity_macro":  float(np.mean(spec_list)),
        "n_samples":          int(len(y_true)),
        "n_classes":          n_classes,
        "threshold":          threshold,
    }


def format_metrics_multilabel(metrics: dict) -> str:
    """Formatea las métricas para logging."""
    lines = [
        f"AUC macro={metrics['auc_macro']:.4f} | "
        f"mAP={metrics['map']:.4f} | "
        f"Acc macro={metrics['accuracy_macro']:.4f}",
        "",
        f"{'Clase':<22} {'AUC':>6} {'AP':>6} {'Sens':>6} {'Spec':>6}",
        "-" * 50,
    ]
    for i, cls in enumerate(CLASSES_14):
        auc  = metrics["auc_per_class"][i]
        ap   = metrics["ap_per_class"][i]
        sens = metrics["sensitivity_per_class"][i]
        spec = metrics["specificity_per_class"][i]
        auc_str  = f"{auc:.4f}"  if not np.isnan(auc) else "  N/A"
        ap_str   = f"{ap:.4f}"   if not np.isnan(ap)  else "  N/A"
        lines.append(
            f"{cls:<22} {auc_str:>6} {ap_str:>6} "
            f"{sens:>6.4f} {spec:>6.4f}"
        )
    return "\n".join(lines)


def _to_numpy(x) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


# ─── Smoke test ───────────────────────────────────────────────────
if __name__ == "__main__":
    np.random.seed(42)
    N, C = 200, 14
    y_true  = (np.random.rand(N, C) > 0.85).astype(np.float32)
    y_probs = np.clip(y_true + np.random.randn(N, C) * 0.3, 0, 1)

    m = compute_metrics_multilabel(y_true, y_probs)
    print("Smoke test metrics_multilabel.py:")
    print(format_metrics_multilabel(m))
    print(f"\nAUC macro: {m['auc_macro']:.4f}")
    print(f"mAP:       {m['map']:.4f}")
'''
p = Path(f'{CODE_DIR}/src/evaluation/metrics_multilabel.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
from pathlib import Path
content = r'''"""
src/training/train_multilabel.py

Loop de entrenamiento para Sprint 4 multi-label (14 clases NIH).

Diferencias clave respecto a train_s4.py (4 clases):
  - BCEWithLogitsLoss en vez de CrossEntropyLoss.
  - pos_weight para compensar el fuerte desbalance de clases.
  - Sigmoid (no softmax) en la evaluación.
  - Métricas: AUC por clase + mAP (no AUC macro de multi-clase).
  - Misma estrategia de 2 fases que Sprint 3.
"""
from __future__ import annotations

import copy
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader

from src.evaluation.metrics_multilabel import (
    compute_metrics_multilabel,
    format_metrics_multilabel,
)
from src.training.utils import save_checkpoint


# ─── Una epoch de entrenamiento ──────────────────────────────────
def train_one_epoch_ml(model, loader, optimizer, criterion,
                       device, use_amp, scaler):
    model.train()
    losses, all_probs, all_labels = [], [], []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)  # (B, 14)

        optimizer.zero_grad(set_to_none=True)

        if use_amp and scaler:
            with autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)           # (B, 14)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        losses.append(loss.item())
        with torch.no_grad():
            probs = torch.sigmoid(logits.detach().float())
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    metrics    = compute_metrics_multilabel(all_labels, all_probs)
    return {"loss": float(np.mean(losses)), "metrics": metrics}


# ─── Una epoch de validación ──────────────────────────────────────
@torch.no_grad()
def validate_one_epoch_ml(model, loader, criterion, device):
    model.eval()
    losses, all_probs, all_labels = [], [], []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        logits = model(images)
        losses.append(criterion(logits, labels).item())
        probs = torch.sigmoid(logits.float())
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    metrics    = compute_metrics_multilabel(all_labels, all_probs)
    return {"loss": float(np.mean(losses)), "metrics": metrics}


# ─── Loop de una fase ─────────────────────────────────────────────
def run_phase_ml(model, train_loader, val_loader, optimizer,
                 criterion, scheduler, epochs, patience,
                 device, use_amp, logger, phase_name, ckpt_path,
                 start_auc=-float("inf")):

    scaler       = GradScaler(device="cuda") if use_amp else None
    best_auc     = start_auc
    best_state   = None
    no_improve   = 0
    history      = []

    logger.info("=" * 70)
    logger.info("FASE %s | %d epochs | patience=%d", phase_name, epochs, patience)
    logger.info("Entrenables: %d", model.trainable_params())
    logger.info("=" * 70)

    for epoch in range(1, epochs + 1):
        lr = optimizer.param_groups[0]["lr"]
        tr = train_one_epoch_ml(model, train_loader, optimizer,
                                criterion, device, use_amp, scaler)
        va = validate_one_epoch_ml(model, val_loader, criterion, device)

        if scheduler:
            scheduler.step()

        logger.info(
            "[%s] Epoch %2d/%d | LR=%.2e | "
            "train loss=%.4f AUC=%.4f mAP=%.4f | "
            "val loss=%.4f AUC=%.4f mAP=%.4f",
            phase_name, epoch, epochs, lr,
            tr["loss"], tr["metrics"]["auc_macro"], tr["metrics"]["map"],
            va["loss"], va["metrics"]["auc_macro"], va["metrics"]["map"],
        )

        # Log AUC por clase en val (cada 5 epochs para no saturar el log)
        if epoch % 5 == 0 or epoch == 1:
            for i, auc in enumerate(va["metrics"]["auc_per_class"]):
                if not np.isnan(auc):
                    from src.datasets.nih_multilabel import CLASSES_14
                    logger.info("  val AUC %s: %.4f",
                                CLASSES_14[i], auc)

        history.append({
            "phase": phase_name, "epoch": epoch, "lr": lr,
            "train_loss": tr["loss"], "train_metrics": tr["metrics"],
            "val_loss":   va["loss"], "val_metrics":   va["metrics"],
        })

        score = va["metrics"]["auc_macro"]
        if score > best_auc:
            best_auc   = score
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            torch.save({
                "phase": phase_name, "epoch": epoch,
                "model_state_dict": best_state,
                "best_val_auc_macro": best_auc,
                "best_val_map": va["metrics"]["map"],
                "num_classes": 14,
            }, ckpt_path)
            logger.info("  → NUEVO MEJOR: AUC_macro=%.4f mAP=%.4f",
                        best_auc, va["metrics"]["map"])
        else:
            no_improve += 1
            if no_improve >= patience:
                logger.info("  Early stopping (patience=%d).", patience)
                break

    logger.info("Fase %s terminada. Mejor AUC_macro=%.4f",
                phase_name, best_auc)
    return best_state, history, best_auc


# ─── Entrenamiento principal ──────────────────────────────────────
def train_model_multilabel(model, train_loader, val_loader,
                            cfg, device, logger) -> dict:
    """
    Entrena el CNN-ViT multi-label en 2 fases.
    Misma estrategia que Sprint 3 pero con BCE loss.
    """
    ckpt_dir = Path(cfg.paths["checkpoints"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    use_amp = cfg.training.get("use_amp", False) and torch.cuda.is_available()

    # pos_weight para BCEWithLogitsLoss
    if cfg.training.get("use_pos_weight", True):
        logger.info("Calculando pos_weight desde train set...")
        pos_weight = train_loader.dataset.compute_pos_weight(device)
        logger.info("pos_weight (min=%.2f, max=%.2f, mean=%.2f)",
                    pos_weight.min().item(),
                    pos_weight.max().item(),
                    pos_weight.mean().item())
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    p1 = cfg.training["phase1"]
    p2 = cfg.training["phase2"]
    all_history = []

    # ─── FASE 1: CNN congelada ────────────────────────────────────
    logger.info("Iniciando Fase 1: CNN congelada, ViT entrena")
    model.set_phase(1)

    opt1 = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=p1["learning_rate"],
        weight_decay=p1.get("weight_decay", 1e-4),
    )
    sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt1, T_max=p1["epochs"], eta_min=1e-6
    )

    bs1, h1, auc_p1 = run_phase_ml(
        model, train_loader, val_loader, opt1, criterion, sched1,
        epochs=p1["epochs"], patience=p1.get("patience", 5),
        device=device, use_amp=use_amp, logger=logger,
        phase_name="P1",
        ckpt_path=ckpt_dir / "sprint4ml_phase1_best.pt",
    )
    all_history.extend(h1)
    if bs1:
        model.load_state_dict(bs1)

    # ─── FASE 2: Full FT con LR diferenciado ─────────────────────
    logger.info("Iniciando Fase 2: full fine-tuning con LR diferenciado")
    model.set_phase(2)

    param_groups = model.get_param_groups(
        lr_cnn=p2["lr_cnn"],
        lr_vit=p2["lr_vit"],
    )
    opt2 = torch.optim.Adam(param_groups,
                             weight_decay=p2.get("weight_decay", 1e-4))

    sched_cfg = p2.get("scheduler", {})
    sched2 = None
    if sched_cfg.get("enabled", False):
        sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt2,
            T_max=sched_cfg.get("T_max", p2["epochs"]),
            eta_min=sched_cfg.get("eta_min", 1e-7),
        )

    bs2, h2, auc_p2 = run_phase_ml(
        model, train_loader, val_loader, opt2, criterion, sched2,
        epochs=p2["epochs"], patience=p2.get("patience", 10),
        device=device, use_amp=use_amp, logger=logger,
        phase_name="P2",
        ckpt_path=ckpt_dir / "sprint4ml_phase2_best.pt",
        start_auc=auc_p1,
    )
    all_history.extend(h2)

    best_p2_map = max(
        h["val_metrics"]["map"] for h in h2
        if not np.isnan(h["val_metrics"]["map"])
    ) if h2 else 0.0

    logger.info("=" * 70)
    logger.info("ENTRENAMIENTO MULTI-LABEL TERMINADO")
    logger.info("  Fase 1 best AUC_macro: %.4f", auc_p1)
    logger.info("  Fase 2 best AUC_macro: %.4f", auc_p2)
    logger.info("  Fase 2 best mAP:       %.4f", best_p2_map)
    logger.info("=" * 70)

    return {
        "history":             all_history,
        "best_val_auc_phase1": float(auc_p1),
        "best_val_auc_phase2": float(auc_p2),
        "best_val_map_phase2": float(best_p2_map),
        "checkpoint_path":     str(ckpt_dir / "sprint4ml_phase2_best.pt"),
        "num_classes":         14,
    }
'''
p = Path(f'{CODE_DIR}/src/training/train_multilabel.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
from pathlib import Path
content = r'''"""
scripts/prepare_splits_multilabel.py

Genera splits train/val/test para clasificación multi-label (14 clases).

Diferencias respecto a prepare_splits_s4.py (4 clases):
  - NO hay balance de clases (todas las imágenes se usan).
  - El label de cada imagen es un vector binario de 14 posiciones.
  - Split estratificado por presencia/ausencia de la clase más rara.
  - Train: TODAS las imágenes del paciente seleccionado.
  - Val/Test: 1 imagen/paciente.

Con 102,120 imágenes disponibles, train será ~70,000+ imágenes.
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.training.utils import set_seed, get_logger, load_config

CLASSES_14 = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion",    "Emphysema",    "Fibrosis",      "Hernia",
    "Infiltration","Mass",         "Nodule",         "Pleural_Thickening",
    "Pneumonia",   "Pneumothorax",
]


def parse_multilabel(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega columnas binarias para cada una de las 14 clases."""
    df = df.copy()
    for cls in CLASSES_14:
        df[cls] = df["Finding Labels"].str.contains(cls).astype(int)
    df["is_no_finding"] = (df["Finding Labels"] == "No Finding").astype(int)
    return df


def filter_views(df: pd.DataFrame, views: list) -> pd.DataFrame:
    return df[df["View Position"].isin(views)].copy()


def get_patient_stratification_label(df: pd.DataFrame) -> pd.Series:
    """
    Para estratificar el split, usamos si el paciente tiene
    al menos una patología (1) o no (0).
    Esto mantiene la proporción de casos positivos/negativos.
    """
    def patient_label(group):
        # 1 si tiene alguna patología, 0 si es puro "No Finding"
        return int((group["is_no_finding"] == 0).any())

    return df.groupby("Patient ID").apply(patient_label)


def stratified_split_patients(patient_labels: pd.Series,
                               train_ratio: float,
                               val_ratio: float,
                               test_ratio: float,
                               seed: int,
                               logger) -> tuple[set, set, set]:
    X = patient_labels.index.to_numpy()
    y = patient_labels.values

    # Separar test
    n_splits_test = max(2, round(1.0 / test_ratio))
    skf1 = StratifiedKFold(n_splits=n_splits_test, shuffle=True, random_state=seed)
    tv_idx, test_idx = next(skf1.split(X, y))

    test_pats = set(X[test_idx])
    X_tv, y_tv = X[tv_idx], y[tv_idx]

    # Separar val
    val_rel = val_ratio / (train_ratio + val_ratio)
    n_splits_val = max(2, round(1.0 / val_rel))
    skf2 = StratifiedKFold(n_splits=n_splits_val, shuffle=True, random_state=seed)
    tr_idx, val_idx = next(skf2.split(X_tv, y_tv))

    train_pats = set(X_tv[tr_idx])
    val_pats   = set(X_tv[val_idx])

    logger.info("Split: train=%d, val=%d, test=%d pacientes",
                len(train_pats), len(val_pats), len(test_pats))
    return train_pats, val_pats, test_pats


def sample_one_per_patient(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    rng  = np.random.default_rng(seed)
    rows = []
    for _, group in df.groupby("Patient ID"):
        idx = rng.integers(0, len(group))
        rows.append(group.iloc[idx])
    return pd.DataFrame(rows).reset_index(drop=True)


def report_split(name: str, df: pd.DataFrame, logger) -> None:
    n = len(df)
    n_pats = df["Patient ID"].nunique()
    n_nf   = (df["is_no_finding"] == 1).sum()
    logger.info("%s: %d imgs | %d pacientes | No Finding=%d (%.1f%%)",
                name, n, n_pats, n_nf, n_nf/n*100)
    for cls in CLASSES_14:
        count = df[cls].sum()
        logger.info("  %s: %d (%.1f%%)", cls, count, count/n*100)


def main(config_path=None):
    if config_path is None:
        config_path = CODE_DIR / "configs" / "sprint4_multilabel.yaml"
    cfg = load_config(config_path)
    set_seed(cfg.project["seed"])

    log_file = Path(cfg.paths["logs"]) / "prepare_splits_ml.log"
    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("prepare_splits_ml", log_file=log_file)

    logger.info("=" * 65)
    logger.info("PREPARE SPLITS Multi-label (14 clases NIH)")
    logger.info("=" * 65)

    # 1. Cargar CSV
    subset_csv = Path(cfg.paths["data_raw"]) / "Data_Entry_subset_local.csv"
    df = pd.read_csv(subset_csv)
    logger.info("CSV cargado: %d filas", len(df))

    # 2. Filtrar vistas
    df = filter_views(df, cfg.dataset["views"])
    logger.info("Tras filtrar vistas PA/AP: %d filas", len(df))

    # 3. Parsear labels multi-label
    df = parse_multilabel(df)

    # Estadísticas
    logger.info("Distribución de las 14 clases:")
    for cls in CLASSES_14:
        n   = df[cls].sum()
        n_p = df[df[cls] == 1]["Patient ID"].nunique()
        logger.info("  %-22s: %6d imgs | %5d pacientes", cls, n, n_p)
    logger.info("  No Finding: %d imgs", df["is_no_finding"].sum())

    # 4. Split estratificado por paciente
    patient_labels = get_patient_stratification_label(df)
    logger.info("Total pacientes únicos: %d", len(patient_labels))

    train_pats, val_pats, test_pats = stratified_split_patients(
        patient_labels,
        cfg.splits["train_ratio"],
        cfg.splits["val_ratio"],
        cfg.splits["test_ratio"],
        seed=cfg.project["seed"],
        logger=logger,
    )

    # Verificar no leakage
    assert not (train_pats & val_pats),  "Leakage train-val"
    assert not (train_pats & test_pats), "Leakage train-test"
    assert not (val_pats   & test_pats), "Leakage val-test"
    logger.info("Sin leakage entre splits.")

    # 5. Construir DataFrames
    # Train: TODAS las imágenes del paciente
    df_train = df[df["Patient ID"].isin(train_pats)].reset_index(drop=True)

    # Val/Test: 1 imagen/paciente
    df_val  = sample_one_per_patient(
        df[df["Patient ID"].isin(val_pats)],
        seed=cfg.project["seed"] + 1
    )
    df_test = sample_one_per_patient(
        df[df["Patient ID"].isin(test_pats)],
        seed=cfg.project["seed"] + 2
    )

    # 6. Reportar
    logger.info("-" * 65)
    report_split("TRAIN", df_train, logger)
    report_split("VAL",   df_val,   logger)
    report_split("TEST",  df_test,  logger)

    # 7. Guardar
    processed_dir = Path(cfg.paths["data_processed"])
    processed_dir.mkdir(parents=True, exist_ok=True)

    # Columnas a guardar
    base_cols = ["Image Index", "Finding Labels", "Patient ID",
                 "View Position", "is_no_finding"]
    all_cols  = base_cols + CLASSES_14

    df_train[all_cols].to_csv(processed_dir / "train.csv", index=False)
    df_val[all_cols].to_csv(processed_dir   / "val.csv",   index=False)
    df_test[all_cols].to_csv(processed_dir  / "test.csv",  index=False)

    logger.info("-" * 65)
    logger.info("Splits guardados en: %s", processed_dir)
    logger.info("  train.csv: %d filas", len(df_train))
    logger.info("  val.csv:   %d filas", len(df_val))
    logger.info("  test.csv:  %d filas", len(df_test))
    logger.info("  Ratio train/S2: %.1fx", len(df_train) / 739)
    logger.info("=" * 65)
    logger.info("PREPARE SPLITS ML OK")


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/prepare_splits_multilabel.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
from pathlib import Path
content = r'''"""
scripts/run_train_multilabel.py
Entrypoint Sprint 4 multi-label — 14 clases NIH.
"""
from __future__ import annotations
import json, os, sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih_multilabel import build_nih_multilabel_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.cnn_vit import build_cnn_vit
from src.training.train_multilabel import train_model_multilabel
from src.training.utils import get_device, get_logger, load_config, set_seed


def main():
    h5_path       = os.environ.get("H5_PATH",       "/content/nih_images.h5")
    processed_dir = os.environ.get("PROCESSED_DIR", "/content/processed_s4ml")

    cfg    = load_config(CODE_DIR / "configs" / "sprint4_multilabel.yaml")
    set_seed(cfg.project["seed"])

    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("train_ml",
                        log_file=Path(cfg.paths["logs"]) / "train_ml.log")

    logger.info("=" * 70)
    logger.info("SPRINT 4 MULTI-LABEL — 14 clases NIH | CNN-ViT 2 fases")
    logger.info("=" * 70)

    device = get_device()
    logger.info("Device: %s%s", device,
                f" | {torch.cuda.get_device_name(0)}" if device.type=="cuda" else "")

    train_tfm = build_train_transform(cfg.preprocessing["image_size"])
    eval_tfm  = build_eval_transform(cfg.preprocessing["image_size"])

    train_ds, val_ds, test_ds = build_nih_multilabel_datasets(
        processed_dir, h5_path, train_tfm, eval_tfm
    )
    logger.info("Train: %d | Val: %d | Test: %d",
                len(train_ds), len(val_ds), len(test_ds))

    dist = train_ds.class_distribution()
    for cls, count in dist.items():
        logger.info("  %s: %d imgs positivas", cls, count)

    nw = min(cfg.training.get("num_workers", 2), 4)
    train_loader = DataLoader(train_ds, batch_size=cfg.training["batch_size"],
                              shuffle=True, num_workers=nw,
                              pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=cfg.training["batch_size"],
                              shuffle=False, num_workers=nw,
                              pin_memory=(device.type=="cuda"))

    model = build_cnn_vit(cfg).to(device)
    logger.info("Modelo CNN-ViT multi-label: %d clases | %d params",
                cfg.model["num_classes"], model.total_params())

    result = train_model_multilabel(model, train_loader, val_loader,
                                     cfg, device, logger)

    hp = Path(cfg.paths["experiments"]) / "history_s4ml.json"
    hp.parent.mkdir(parents=True, exist_ok=True)
    with hp.open("w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    logger.info("History: %s", hp)
    logger.info("RESUMEN FINAL:")
    logger.info("  P1 AUC: %.4f | P2 AUC: %.4f | P2 mAP: %.4f",
                result["best_val_auc_phase1"],
                result["best_val_auc_phase2"],
                result["best_val_map_phase2"])


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/run_train_multilabel.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
content = r'''"""
scripts/run_train_multilabel.py
Entrypoint Sprint 4 multi-label — 14 clases NIH.
"""
from __future__ import annotations
import json, os, sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih_multilabel import build_nih_multilabel_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.cnn_vit import build_cnn_vit
from src.training.train_multilabel import train_model_multilabel
from src.training.utils import get_device, get_logger, load_config, set_seed


def main():
    h5_path       = os.environ.get("H5_PATH",       "/content/nih_images.h5")
    processed_dir = os.environ.get("PROCESSED_DIR", "/content/processed_s4ml")

    cfg    = load_config(CODE_DIR / "configs" / "sprint4_multilabel.yaml")
    set_seed(cfg.project["seed"])

    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("train_ml",
                        log_file=Path(cfg.paths["logs"]) / "train_ml.log")

    logger.info("=" * 70)
    logger.info("SPRINT 4 MULTI-LABEL — 14 clases NIH | CNN-ViT 2 fases")
    logger.info("=" * 70)

    device = get_device()
    logger.info("Device: %s%s", device,
                f" | {torch.cuda.get_device_name(0)}" if device.type=="cuda" else "")

    train_tfm = build_train_transform(cfg.preprocessing["image_size"])
    eval_tfm  = build_eval_transform(cfg.preprocessing["image_size"])

    train_ds, val_ds, test_ds = build_nih_multilabel_datasets(
        processed_dir, h5_path, train_tfm, eval_tfm
    )
    logger.info("Train: %d | Val: %d | Test: %d",
                len(train_ds), len(val_ds), len(test_ds))

    dist = train_ds.class_distribution()
    for cls, count in dist.items():
        logger.info("  %s: %d imgs positivas", cls, count)

    nw = min(cfg.training.get("num_workers", 2), 4)
    train_loader = DataLoader(train_ds, batch_size=cfg.training["batch_size"],
                              shuffle=True, num_workers=nw,
                              pin_memory=(device.type=="cuda"))
    val_loader   = DataLoader(val_ds,   batch_size=cfg.training["batch_size"],
                              shuffle=False, num_workers=nw,
                              pin_memory=(device.type=="cuda"))

    model = build_cnn_vit(cfg).to(device)
    logger.info("Modelo CNN-ViT multi-label: %d clases | %d params",
                cfg.model["num_classes"], model.total_params())

    result = train_model_multilabel(model, train_loader, val_loader,
                                     cfg, device, logger)

    hp = Path(cfg.paths["experiments"]) / "history_s4ml.json"
    hp.parent.mkdir(parents=True, exist_ok=True)
    with hp.open("w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    logger.info("History: %s", hp)
    logger.info("RESUMEN FINAL:")
    logger.info("  P1 AUC: %.4f | P2 AUC: %.4f | P2 mAP: %.4f",
                result["best_val_auc_phase1"],
                result["best_val_auc_phase2"],
                result["best_val_map_phase2"])


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/run_train_multilabel.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


## 4. Generar splits multi-label

In [ ]:
import subprocess

result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/prepare_splits_multilabel.py'],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr[-4000:])


In [ ]:
import pandas as pd

os.makedirs(PROCESSED_ML, exist_ok=True)
for csv in ['train.csv', 'val.csv', 'test.csv']:
    shutil.copy(f'{PROJECT_ROOT}/data/processed_s4ml/{csv}',
                f'{PROCESSED_ML}/{csv}')
    df = pd.read_csv(f'{PROCESSED_ML}/{csv}')
    print(f'{csv}: {len(df)} imgs | {df["Patient ID"].nunique()} pacientes')
    # Distribución de clases
    for cls in CLASS_NAMES_14[:4]:  # primeras 4 para no saturar
        print(f'  {cls}: {df[cls].sum()} positivos ({df[cls].mean()*100:.1f}%)')


## 5. Smoke test: vector multi-label

In [ ]:
for mod in list(__import__('sys').modules.keys()):
    if mod.startswith('src.'): del __import__('sys').modules[mod]

from src.datasets.nih_multilabel import (
    NIHDatasetMultiLabel, CLASSES_14, findings_to_vector
)
from src.datasets.transforms import build_eval_transform

# Test de conversión de labels
print('Test findings_to_vector:')
casos = ["No Finding", "Cardiomegaly", "Cardiomegaly|Effusion", "Infiltration|Pneumonia"]
for caso in casos:
    vec = findings_to_vector(caso)
    activas = [CLASSES_14[i] for i in range(14) if vec[i] == 1]
    print(f'  "{caso}"')
    print(f'    → activas: {activas if activas else ["ninguna"]}')

print()

# Dataset
eval_tfm = build_eval_transform(224)
train_ds = NIHDatasetMultiLabel(f'{PROCESSED_ML}/train.csv', H5_LOCAL, eval_tfm)
val_ds   = NIHDatasetMultiLabel(f'{PROCESSED_ML}/val.csv',   H5_LOCAL, eval_tfm)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')
print()

# Distribución
dist = train_ds.class_distribution()
print('Distribución de clases en train (ordenada por frecuencia):')
for cls, count in sorted(dist.items(), key=lambda x: -x[1]):
    pct = count/len(train_ds)*100
    print(f'  {cls:<22}: {count:>6,} ({pct:4.1f}%)')
print(f'  No Finding: {train_ds.no_finding_count():,} imágenes')

# Sample
sample = train_ds[0]
print(f'\nMuestra[0]:')
print(f'  shape: {sample["image"].shape}')
print(f'  label: {sample["label"].tolist()[:6]}... (primeras 6 de 14)')
activas = [CLASSES_14[i] for i in range(14) if sample["label"][i] > 0.5]
print(f'  hallazgos: {activas if activas else ["No Finding"]}')
print(f'  imagen: {sample["image_index"]}')
print('✓ Dataset multi-label OK')


## 6. Entrenar CNN-ViT Multi-label

**Tiempo estimado: ~3-4 horas en T4.**

El log muestra por epoch:
- `AUC` = AUC macro (promedio de las 14 clases)
- `mAP` = mean Average Precision (métrica principal multi-label)

Cada 5 epochs muestra el AUC por cada clase individualmente.


In [ ]:
import subprocess, os, sys

os.makedirs(f'{PROJECT_ROOT}/experiments_s4ml/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments_s4ml/logs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments_s4ml/figures', exist_ok=True)

env = {
    **os.environ,
    'H5_PATH':       H5_LOCAL,
    'PROCESSED_DIR': PROCESSED_ML,
}

print('Iniciando entrenamiento multi-label (14 clases)...')
print('Los logs aparecen en tiempo real:\n')

# Usar Popen en vez de run() para ver output en tiempo real
process = subprocess.Popen(
    ['python', f'{CODE_DIR}/scripts/run_train_multilabel.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # combinar stdout y stderr
    text=True,
    env=env,
    bufsize=1,                  # line-buffered
)

# Imprimir cada línea en tiempo real
for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\nReturn code: {process.returncode}')


## 7. Resultados y comparativa

In [ ]:
import json, matplotlib.pyplot as plt
import numpy as np

h_ml_path = f'{PROJECT_ROOT}/experiments_s4ml/history_s4ml.json'
h_s2_path = f'{PROJECT_ROOT}/experiments_s2/history_s2.json'
h_s4_path = f'{PROJECT_ROOT}/experiments_s4/history_s4.json'

with open(h_ml_path) as f: d_ml = json.load(f)

hist_ml = d_ml['history']
p2_hist = [h for h in hist_ml if h.get('phase') == 'P2']
best_ml = max(p2_hist, key=lambda h: h['val_metrics']['auc_macro'])['val_metrics']

print('RESULTADOS SPRINT 4 MULTI-LABEL (val):')
print('='*55)
print(f"AUC macro: {best_ml['auc_macro']:.4f}")
print(f"mAP:       {best_ml['map']:.4f}")
print()
print(f"{'Clase':<22} {'AUC':>6} {'AP':>6}")
print('-'*38)
for i, cls in enumerate(CLASS_NAMES_14):
    auc = best_ml['auc_per_class'][i]
    ap  = best_ml['ap_per_class'][i]
    auc_str = f'{auc:.4f}' if not np.isnan(auc) else '  N/A'
    ap_str  = f'{ap:.4f}'  if not np.isnan(ap)  else '  N/A'
    print(f'{cls:<22} {auc_str:>6} {ap_str:>6}')

# Comparativa con Wang et al. 2017 (referencia)
WANG_AUC = {
    'Atelectasis': 0.7003, 'Cardiomegaly': 0.8100,
    'Consolidation': 0.7032, 'Edema': 0.8052,
    'Effusion': 0.7585, 'Emphysema': 0.8330,
    'Fibrosis': 0.7859, 'Hernia': 0.8717,
    'Infiltration': 0.6614, 'Mass': 0.6931,
    'Nodule': 0.6693, 'Pleural_Thickening': 0.6835,
    'Pneumonia': 0.6580, 'Pneumothorax': 0.7993,
}

print()
print('COMPARATIVA vs Wang et al. 2017:')
print(f"{'Clase':<22} {'Wang17':>7} {'Sprint4':>8} {'Delta':>7}")
print('-'*48)
deltas = []
for i, cls in enumerate(CLASS_NAMES_14):
    auc_s4   = best_ml['auc_per_class'][i]
    auc_wang = WANG_AUC.get(cls, 0)
    if not np.isnan(auc_s4):
        delta = auc_s4 - auc_wang
        deltas.append(delta)
        arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
        print(f'{cls:<22} {auc_wang:>7.4f} {auc_s4:>8.4f} {delta:>+7.4f} {arrow}')

if deltas:
    print(f'\nDelta promedio vs Wang17: {np.mean(deltas):+.4f}')


## 8. Siguiente paso

Con los resultados del Sprint 4 multi-label:
1. Evaluar sobre **test set** con el checkpoint final.
2. Documentar Sprint 4.
3. Actualizar el backend para soportar 14 clases (cambio de YAML).
